# 04.2 — A small validated client

[Class index and setup](README.md) · [Previous: 04.1](04.1-frontend-backend-and-http.ipynb) · [Next: 04.3](04.3-failures-and-ai-api-previews.ipynb)

**Purpose:** Replace the saved input boundary with one small HTTP client.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's separate project is additional work.

## Learning outcomes

- Use Requests with params and an explicit timeout.
- Check status before decoding and validate before calculating.
- Preserve the Observation and Report interfaces from earlier classes.

## Starting point

You know basic Python, functions, JSON-like dictionaries, and the earlier Observation/Report idea. The complete package and fixtures are supplied again; no previous notebook or exercise answer is needed. Open the supplied class workspace in VS Code, select this class's `.venv`, then **Restart Kernel → Run All**.

Default cells inspect saved responses and run offline checks. They do not start a server, call an external API, read credentials, install software, or download models. Live commands are explicit **manual terminal steps**. An HTTP-shaped fixture is not evidence of a live request. Empty answer cells are safe.

Attempt exercises before using the separate `%load` cell: uncomment its last line and run to reveal, then run the populated cell separately. A loaded solution replaces its loader cell; restarting the kernel does not restore that source. Keep your attempt in its own cell.

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/api.py").is_file() or not Path("fixtures/observations.json").is_file():
    raise FileNotFoundError(
        "Open 04-consuming-apis.code-workspace and restart the kernel. "
        "Download the complete class folder, including report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available; default execution is offline.")

## In class · 30–50 minutes — Four steps are enough

The core client does four jobs in order:

1. **Request:** send `GET`, query parameters, and a timeout.
2. **Check:** call `raise_for_status()` to reject HTTP error statuses.
3. **Decode:** use `json()` to turn the body into a Python object.
4. **Validate:** check that it is a list of records with our expected fields.

The following is a **manual live example**, not an automatically executed cell:

```python
import requests
from report_app.api import LOCAL_URL, parse_observations

response = requests.get(LOCAL_URL, params={"label": "north"}, timeout=5)
response.raise_for_status()
records = parse_observations(response.json())
```

`params=` handles query encoding. `timeout=5` limits connection/read waiting; it is not a guaranteed five-second deadline for the entire operation. A successful JSON decode does not validate the application's data contract. Python type hints also do not validate an incoming dictionary by themselves.

### Keep the internal model stable

The supplied `Observation` still has `label` and `value`. A value must be finite and nonnegative; a label must be a nonempty string. Pydantic's ordinary numeric coercion still applies, so a numeric string may become a float. No new identifiers or category fields are required in `Report`.

`parse_observations` validates the whole list. One malformed row rejects the load; we do not silently skip it and return a misleading partial summary.

Optional illustration: [HTTP response parts and teaching caveats](IMAGE_SUGGESTIONS.md). Use it only to locate status, headers, and body; Requests handles protocol framing.

In [ ]:
import inspect
from report_app.api import parse_observations
from report_app.analysis import summarize
from report_app.source import read_fixture

print(inspect.getsource(parse_observations))
records = parse_observations(read_fixture("observations.json"))
print(type(records[0]).__name__, records[0].model_dump())
print(summarize(records))
assert summarize(records) == {"count": 2, "total": 6, "mean": 3}

### Inspect the reusable acquisition boundary

[api.py](report_app/api.py) keeps error explanations in `get_json` and the record contract in `parse_observations`. `fetch_observations` joins them. There is no API-client class, plugin system, asynchronous code, or retry framework to understand. We will read the small exception branches in the next part.

In [ ]:
from report_app.api import fetch_observations, get_json

print(inspect.getsource(get_json))
print(inspect.getsource(fetch_observations))

### Exercise the client offline

We can replace only the external GET and still execute the real status check, JSON decode, validation, and summary. `patch` temporarily replaces the function used by the client and restores it when the `with` block ends. This continues the controlled-boundary idea from class 03.

The supplied `saved_response` helper constructs a Requests response from a small fixture. You do not need to implement that test setup. It does not run the server and it does not automatically filter records according to parameters; select the matching fixture deliberately.

In [ ]:
from unittest.mock import patch
from examples.saved_http import saved_response

with patch("report_app.api.requests.get", return_value=saved_response("north.json")) as get:
    north_records = fetch_observations(label="north")
    get.assert_called_once_with(
        "http://127.0.0.1:8000/observations", params={"label": "north"}, timeout=5
    )
print("OFFLINE client result:", summarize(north_records))

### 🧩 Exercise 04.2.E1 — Write the four-step client

> **In class · 10 minutes · 50–60**
>
> **Goal:** Acquire one validated observation without moving HTTP into Report.
>
> **Start here:** LOCAL_URL, parse_observations, saved_response("north.json"), and the with patch example above.
>
> **Task:** Write fetch_north(): request label=north with timeout=5, check status, decode JSON, and return validated observations. Run it under the supplied replacement GET. Pass its result to summarize.
>
> **Check:** The result contains an Observation and its summary is count 1, total 2.5, mean 2.5. Your function still works with a real GET when the local server is started explicitly.
>
> **Hint:** Keep requests.get → raise_for_status → json → parse_observations in that order. Reuse the supplied validation function.

In [ ]:
# Your attempt for 04.2.E1.
# Record code, observations, and your explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_04_2_e1.py

## Guided follow-up — Independent work · 15 minutes

Spend five minutes tracing `fetch_observations` from GET to summary. Spend five changing a scratch input to a negative value or missing label and inspecting the validation failure. Spend five explaining the boundary: why does receiving JSON not make it a valid Observation? If you inspect a caught error, `exc.__cause__` retains the original validation detail.

**Completion:** show a normal record, a rejected input, and the unchanged summary function. The report must never receive a half-validated result. Returning a list of validated records keeps future web or AI interfaces independent of the HTTP format.

## References

- [Requests: requests, parameters, JSON, status and timeouts](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Pydantic TypeAdapter](https://docs.pydantic.dev/latest/concepts/type_adapter/)
- [Class index](README.md)